A set of simulations & hypothesis tests for assessing the statistical power of "minP vs CRE of interest" tests in the shendure dataset...

Essentially a better version of `power_shendure_vs_minp.ipynb` with a more representative distribution of positive and negative effects. Specifically, we will be using the real values, plus many negatives.

Based on UKBB paper, expect ~30% of library to be active. So we will add 2x original size of negatives... 

We will also reduce computational burden by producing half-orthos.

# Imports & dask cluster creation

In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster
from pathlib import Path

%load_ext autoreload
%autoreload 2

2026-01-30 09:01:17.372351: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-30 09:01:17.376009: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/2024a/software/code-server/4.103.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cUR

In [2]:
local=False
if local:
    cluster=LocalCluster(memory_limit='48G')
    client=Client(cluster)
else:
    cluster=SLURMCluster(
        cores=2,#cores per slurm job
        memory="16G",#memory per slurm job
        processes=1,#dask workers per slurm job,
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=1:00:00",
            f"--output=worker_%j.out"]
    )
    #cluster.scale(jobs=20)
    cluster.scale(jobs=2)
    client = Client(cluster,
            timeout=f"{5*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s"  # Worker heartbeat interval
        )

In [3]:
data_root=Path("/nfs/roberts/project/pi_skr2/shared/tabula_data")

In [4]:
client.dashboard_link

'http://10.18.22.93:8787/status'

# Ground truth creation

We will use parameter estimates averaged across cell-type nd cre models. 

The shendure dataset is characterized by extremely heterogenious transfection, so different sets of CREs are represented in different cell types, likely due to differential clonotype contribution to different cell-types. For this reason, we don't have ground truth values for many combinations of cre_id, cell type. This means that direct simulation runs into problems, since it allows transfection of any cre into any cell type...

See commit `58a65def6964cea9247c15937dd60714489f1750` and 2026-10-23 and 2026-01-29 notes for further discussion.

In [5]:
primordial=scm.ortho.load(client,data_root/"shendure","ortho_primordial_v4")
primordial.compute_model_qc()

We use by_cell_type models, expecting that they will have more robust estimates...

In [6]:
import pandas as pd
import numpy as np

vals=[]
for key in primordial.by_cell_qc.keys():
    working=primordial.by_cell_qc[key]['dat'].reset_index().drop(columns=["mean(umis_mpra_bc)"])
    working["cell_type"]=key
    vals.append(working)
gt_cell_type=pd.concat(vals)

In [7]:
#casting away from sparse, since it's not sparse anymore
gt_cell_type["mu"] = gt_cell_type["mu"].astype(float)

#this doesn't even work
gt_cell_type["cre_id"]=gt_cell_type["cre_id"].astype(str)
gt_cell_type["cell_type"]=gt_cell_type["cell_type"].astype(str)

In [8]:
gt_cell_type

,cre_id,mu,cell_type
0,Bend5_chr4_8168,0.019097,SurfaceEctoderm
1,Bend5_chr4_8174,0.011630,SurfaceEctoderm
2,Bend5_chr4_8175,0.671256,SurfaceEctoderm
3,Bend5_chr4_8179,0.019683,SurfaceEctoderm
4,Bend5_chr4_8199,0.004408,SurfaceEctoderm
...,...,...,...
80,Txndc12_chr4_7978,0.995304,NeuroectodermRostral
81,eef1aP,44.205127,NeuroectodermRostral
82,pgk1P,9.947188,NeuroectodermRostral
83,reference,0.015677,NeuroectodermRostral


In [9]:
combo_counts=gt_cell_type[gt_cell_type["cre_id"]!="reference"].drop(columns=["mu"]).groupby("cell_type").nunique()
max_tfection=max(combo_counts["cre_id"])
combo_counts

,cre_id
cell_type,
Cardiomyocytes,78
EpiblastPrimitiveStreak,184
ExEndodermParietal,177
ExEndodermVisceral,147
Haematoendothelial,90
Mesoderm,176
NeuroectodermBrain,172
NeuroectodermRostral,84
SurfaceEctoderm,138


Next, remove the cell-type-specific reference values.

In [10]:
gt_cell_type=gt_cell_type[gt_cell_type["cre_id"]!="reference"]

Now, we all identifying information, just making generic strength values...

In [11]:
real_means=gt_cell_type.drop(columns=["cre_id","cell_type"])
real_means

,mu
0,0.019097
1,0.011630
2,0.671256
3,0.019683
4,0.004408
...,...
79,0.674647
80,0.995304
81,44.205127
82,9.947188


In [12]:
num_cell_types=len(gt_cell_type["cell_type"].unique())

In [13]:
num_cell_types

10

In [14]:
synth_cre_names=[f"synthcre_{i}" for i in range(0,len(real_means))]

parts=[]
for i in range(0,num_cell_types):
    working=real_means.sample(frac=1)
    working["cre_id"]=synth_cre_names
    working["cell_type"]=f"ct_{i}"
    parts.append(working)

cartesian=pd.concat(parts).sample(frac=1).reset_index(drop=True)
cartesian

,mu,cre_id,cell_type
0,0.038829,synthcre_346,ct_7
1,0.813245,synthcre_447,ct_7
2,0.032332,synthcre_888,ct_7
3,0.032608,synthcre_579,ct_5
4,0.385632,synthcre_864,ct_4
...,...,...,...
14515,0.059827,synthcre_1134,ct_7
14516,0.633817,synthcre_290,ct_1
14517,0.027624,synthcre_18,ct_1
14518,0.060205,synthcre_986,ct_4


In [15]:
#make 70% inactive
minP=scm.SHENDURE_BOUNDS.reference_activity
num_inactive=int(len(cartesian)*0.7)
cartesian.loc[cartesian.index[:num_inactive],"mu"]=minP
cartesian


,mu,cre_id,cell_type
0,0.019311,synthcre_346,ct_7
1,0.019311,synthcre_447,ct_7
2,0.019311,synthcre_888,ct_7
3,0.019311,synthcre_579,ct_5
4,0.019311,synthcre_864,ct_4
...,...,...,...
14515,0.059827,synthcre_1134,ct_7
14516,0.633817,synthcre_290,ct_1
14517,0.027624,synthcre_18,ct_1
14518,0.060205,synthcre_986,ct_4


In [16]:
#now we pick max_tfection CREs to proceed with, because this is the number of unique cre_id 
#found in the the cell type with the most unique CRE ids (reference, or PSC, in shend)

chosen_cre_id=np.random.choice(cartesian["cre_id"].unique(),max_tfection,replace=False)
sampled_nbs=cartesian[cartesian["cre_id"].isin(chosen_cre_id)]
sampled_nbs


,mu,cre_id,cell_type
2,0.019311,synthcre_888,ct_7
9,0.019311,synthcre_290,ct_6
23,0.019311,synthcre_518,ct_7
28,0.019311,synthcre_424,ct_9
29,0.019311,synthcre_880,ct_7
...,...,...,...
14506,0.010794,synthcre_494,ct_7
14510,0.791271,synthcre_1085,ct_2
14513,0.367562,synthcre_1449,ct_0
14516,0.633817,synthcre_290,ct_1


add universal reference CRE

In [17]:
ref_df=pd.DataFrame({"cell_type":[f"ct_{i}" for i in range(0,num_cell_types)]})
ref_df["mu"]=minP
ref_df["cre_id"]="reference"
ref_df


,cell_type,mu,cre_id
0,ct_0,0.019311,reference
1,ct_1,0.019311,reference
2,ct_2,0.019311,reference
3,ct_3,0.019311,reference
4,ct_4,0.019311,reference
5,ct_5,0.019311,reference
6,ct_6,0.019311,reference
7,ct_7,0.019311,reference
8,ct_8,0.019311,reference
9,ct_9,0.019311,reference


Stack & make one of the celltypes reference...

In [18]:
final_gt=pd.concat([ref_df,sampled_nbs])
final_gt["cell_type"]=final_gt["cell_type"].replace({"ct_0":"reference"})
final_gt

,cell_type,mu,cre_id
0,reference,0.019311,reference
1,ct_1,0.019311,reference
2,ct_2,0.019311,reference
3,ct_3,0.019311,reference
4,ct_4,0.019311,reference
...,...,...,...
14506,ct_7,0.010794,synthcre_494
14510,ct_2,0.791271,synthcre_1085
14513,reference,0.367562,synthcre_1449
14516,ct_1,0.633817,synthcre_290


# Creating artificial libraries

In [19]:
libraries=[scm.simulate_library(CREs=final_gt["cre_id"],
                 library_model=scm.SHENDURE_BOUNDS.library_model)
                 for i in range(5)]

# Creating sim

Create some bounds to simulate from. These will be identical to the normal shendure bounds, except that we simplify to just one cell-type...

In [20]:
bound=scm.SHENDURE_BOUNDS.copy()

In [21]:
print(bound.cells_per_cell_type.name)
print(bound.cells_per_cell_type.index.name)
s=bound.cells_per_cell_type
s

cells_per_cell_type
cell_type


cell_type
Cardiomyocytes              680
EpiblastPrimitiveStreak    3445
ExEndodermParietal         4644
ExEndodermVisceral         3238
Haematoendothelial         1079
Mesoderm                   7427
NeuroectodermBrain         7750
NeuroectodermRostral       1757
SurfaceEctoderm            5168
reference                  8201
Name: cells_per_cell_type, dtype: int64

In [22]:
s.index=[f"ct_{i}" for i in range(1,num_cell_types)]+["reference"]
s

ct_1          680
ct_2         3445
ct_3         4644
ct_4         3238
ct_5         1079
ct_6         7427
ct_7         7750
ct_8         1757
ct_9         5168
reference    8201
Name: cells_per_cell_type, dtype: int64

In [23]:
bound.cells_per_cell_type=s

In [24]:
sim=scm.de_novo_simulation(location=data_root,
                            name="twothird_pow_sim_2026-01-30",
                            client=client,
                            libraries=libraries,
                            library_mapping="corresponding",
                            flatten_overtransfection=False,
                            n_sims=5,
                            experiment_bounds=bound,
                            ground_truth=final_gt)

scMPRAforge: INFO: No 'state.parquet' found for 'twothird_pow_sim_2026-01-30'. Initalizing new object.


In [25]:
sim.gamut()

In [ ]:
sim.save()

In [ ]:
x=scm.scMPRA_data.from_parquet(data_root/"twothird_pow_sim_2026-01-30/simulated_scmpra/4.scmpra")
x.ortho_filter()
x.operations

In [ ]:
client.close()
cluster.close()

# Fit orthos

In [ ]:
sim=scm.de_novo_simulation(location=data_root,
                            name="twothird_pow_sim_2026-01-29",
                            client=client)

In [ ]:
any(sim.ground_truth["cre_id"].unique()=="reference")

In [ ]:
sim.fit_orthos(serial_orthos=True)

In [ ]:
sim

In [ ]:
x.operations

In [ ]:
sim.save()

In [ ]:
print("DONE")

# Wald precompute

In [ ]:
sim=scm.de_novo_simulation(location=data_root,
                            name="twothird_pow_sim_2026-01-26",
                            client=client)

In [ ]:
sim.precompute_wald(cov_method="sandwich")

In [ ]:
sim.precompute_wald(cov_method="opg")

In [ ]:
sim.save()

# Hypothesis testing

## Add the hypotheses...

In [ ]:
example_data=scm.scMPRA_data.from_parquet(sim.scmpradatp/"0.scmpra")

In [ ]:
example_data

In [ ]:
hs_all_ct = scm.make_all_by_celltype_hypotheses(
    counts=example_data,
    reference_cre="reference",
)

In [ ]:
hs_all_ct.df

In [ ]:
sim.add_hypothesis_set("hs_all_ct",hs_all_ct)

## Run tests

In [ ]:
sim=scm.de_novo_simulation(location=data_root,
                            name="twothird_pow_sim_2026-01-26",
                            client=client)

In [ ]:
sim.wald("hs_all_ct")

In [ ]:
sim.wald("hs_all_ct",cov_method="opg")

In [ ]:
sim.mwu("hs_all_ct")

In [ ]:
sim.save()

## Performance metrics

In [ ]:
sim._all_classifier_summary("hs_all_ct")

In [ ]:
sim.performance_barchart("hs_all_ct","auprc")

# Close the cluster

In [ ]:
client.close()
cluster.close()

In [ ]:
#cells_df=scm.load_df_pickle_debug("permerge_cells_df_2a93a57f821d4c9b95cf54fb6a215508.pkl")
##cells_df=scm.cast_string_keys(cells_df,["cell_type", "cre_id"])
#ground_truth=scm.load_df_pickle_debug("premerge_gt_ad527259c2c743cdaed70e6a02e88a0f.pkl")
##ground_truth=scm.cast_string_keys(ground_truth,["cell_type", "cre_id"])

In [ ]:
#cells_df.merge(ground_truth,
#                on=["cell_type","cre_id"],
#                validate="many_to_one",
#                how="left",
#                indicator=True)